In [ ]:
import pandas as pd

# Read the first CSV file into a DataFrame
repairs = pd.read_csv("../../datasets/symmetric_repairs.csv")

repairs

In [ ]:
repairs[(repairs['deleted constraint'] == True)]

In [ ]:
len(repairs[(repairs['deprecated rank'] == True)])

In [ ]:
repairs[(repairs['exception'] == True)]

In [ ]:
import requests
import xml.etree.ElementTree as ET

def isRemovedWithObj(subject, property, obj):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"ASK {{ <{subject}> <{property}> <{obj}> }}"

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)

    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
subject = "http://www.wikidata.org/entity/Q10925338"
property = "http://www.wikidata.org/prop/direct/P1322"
obj = "http://www.wikidata.org/entity/Q24840677"
print(isRemovedWithObj(subject, property, obj))

In [ ]:
repairs['instanceRemoved'] = False

In [ ]:
for index, row in repairs.iterrows():
    # Extract subject and property without the prefix "http://www.wikidata.org/entity/"
    subject = row['subject']
    property = row['property']
    obj = row['object']
    
    if (index % 100000 == 0):
        print(index)
    #break
    
    if (obj.startswith('http')):
        # Call isRemovedWithObj function
        removed = isRemovedWithObj(subject, property, obj)
        if removed is not None:
            # Update instanceRemoved column
            repairs.at[index, 'instanceRemoved'] = removed
        else:
            print("error in:", subject, " ", property, " ", obj)

# Display the updated DataFrame
#print(filtered_df)

In [ ]:
import pandas as pd
import requests
import xml.etree.ElementTree as ET
import os
import time
from tqdm import tqdm

def isRemovedWithObj(subject, property, obj, max_attempts=3):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"ASK {{ <{subject}> <{property}> <{obj}> }}"

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    for attempt in range(1, max_attempts + 1):
        response = requests.get(url, headers=headers)
        
        # Check if the request was successful and parse the response
        if response.ok:
            # Parse the XML response
            root = ET.fromstring(response.text)
            boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
            if boolean_element is not None:
                return boolean_element.text.lower() == 'false'
            else:
                print("Error: 'boolean' element not found in XML response")
                return None
        else:
            print(f"Error on attempt {attempt} for subject: {subject}, property: {property}, object: {obj}")
        
        if attempt < max_attempts:
            time.sleep(1)  # Optional: wait for 1 second before retrying
    
    return None

def process_repairs(repairs, checkpoint_file='symmetric/checkpoint.csv'):
    # Check if there is a checkpoint file and load it
    if os.path.exists(checkpoint_file):
        processed_repair = pd.read_csv(checkpoint_file)
        start_index = len(processed_repair)
        print(f"Checkpoint found. Resuming from index {start_index}.")
    else:
        processed_repair = pd.DataFrame(columns=repairs.columns.tolist())
        start_index = 0

    # Create a progress bar
    with tqdm(total=len(repairs), initial=start_index, desc="Processing repairs") as pbar:
        for index in range(start_index, len(repairs)):
            row = repairs.iloc[index]
            
            subject = row['subject']
            property = row['property']
            obj = row['object']
            
            if (index % 100000 == 0) and (index > start_index):
                print(f"Processing row {index}. Saving checkpoint.")
                # Save checkpoint
                processed_repair.to_csv(checkpoint_file, index=False)
            
            if (index % 10000 == 0):
                print(f"Processed {index} rows.")
            
            if obj.startswith('http'):
                # Call isRemovedWithObj function
                removed = isRemovedWithObj(subject, property, obj)
                if removed is not None:
                    # Update instanceRemoved column in repairs DataFrame
                    repairs.at[index, 'instanceRemoved'] = removed
                    # Create a new row for processed_repair DataFrame
                    new_row = repairs.loc[index].tolist()
                    processed_repair.loc[index] = new_row
                else:
                    print(f"Error in: {subject} {property} {obj}")

            # Update the progress bar
            pbar.update(1)

    # Save the final result
    processed_repair.to_csv(checkpoint_file, index=False)
    print("Processing complete. Final checkpoint saved.")

# Call the function to process the repairs
process_repairs(repairs)

In [ ]:
import pandas as pd
import requests
import xml.etree.ElementTree as ET
import os
import time
from tqdm import tqdm

def isRemovedWithObj(subject, property, obj, max_attempts=3):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"ASK {{ <{subject}> <{property}> <{obj}> }}"

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    for attempt in range(1, max_attempts + 1):
        response = requests.get(url, headers=headers)
        
        # Check if the request was successful and parse the response
        if response.ok:
            # Parse the XML response
            root = ET.fromstring(response.text)
            boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
            if boolean_element is not None:
                return boolean_element.text.lower() == 'false'
            else:
                print("Error: 'boolean' element not found in XML response")
                return None
        else:
            print(f"Error on attempt {attempt} for subject: {subject}, property: {property}, object: {obj}")
        
        if attempt < max_attempts:
            time.sleep(1)  # Optional: wait for 1 second before retrying
    
    return None

def process_repairs(repairs, checkpoint_file='symmetric/checkpoint.csv'):
    # Check if there is a checkpoint file and load it
    if os.path.exists(checkpoint_file):
        processed_repair = pd.read_csv(checkpoint_file)
        start_index = len(processed_repair)
        print(f"Checkpoint found. Resuming from index {start_index}.")
    else:
        processed_repair = pd.DataFrame(columns=repairs.columns.tolist())
        start_index = 0

    # Create a progress bar
    with tqdm(total=len(repairs), initial=start_index, desc="Processing repairs") as pbar:
        for index in range(start_index, len(repairs)):
            row = repairs.iloc[index]
            
            subject = row['subject']
            property = row['property']
            obj = row['object']
            
            if (index % 100000 == 0) and (index > start_index):
                print(f"Processing row {index}. Saving checkpoint.")
                # Save checkpoint
                processed_repair.to_csv(checkpoint_file, index=False)
            
            if (index % 10000 == 0):
                print(f"Processed {index} rows.")
            
            if obj.startswith('http'):
                # Call isRemovedWithObj function
                removed = isRemovedWithObj(subject, property, obj)
                if removed is not None:
                    # Update instanceRemoved column in repairs DataFrame
                    repairs.at[index, 'instanceRemoved'] = removed
                    # Create a new row for processed_repair DataFrame
                    new_row = repairs.loc[index].tolist()
                    processed_repair.loc[index] = new_row
                else:
                    print(f"Error in: {subject} {property} {obj}")

            # Update the progress bar
            pbar.update(1)

    # Save the final result
    processed_repair.to_csv(checkpoint_file, index=False)
    print("Processing complete. Final checkpoint saved.")

# Call the function to process the repairs
process_repairs(repairs)

In [ ]:
import pandas as pd
import requests
import xml.etree.ElementTree as ET
import os
import time
from tqdm import tqdm

def isRemovedWithObj(subject, property, obj, max_attempts=3):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"ASK {{ <{subject}> <{property}> <{obj}> }}"

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    for attempt in range(1, max_attempts + 1):
        response = requests.get(url, headers=headers)
        
        # Check if the request was successful and parse the response
        if response.ok:
            # Parse the XML response
            root = ET.fromstring(response.text)
            boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
            if boolean_element is not None:
                return boolean_element.text.lower() == 'false'
            else:
                print("Error: 'boolean' element not found in XML response")
                return None
        else:
            print(f"Error on attempt {attempt} for subject: {subject}, property: {property}, object: {obj}")
        
        if attempt < max_attempts:
            time.sleep(1)  # Optional: wait for 1 second before retrying
    
    return None

def process_repairs(repairs, checkpoint_file='symmetric/checkpoint.csv'):
    # Check if there is a checkpoint file and load it
    if os.path.exists(checkpoint_file):
        processed_repair = pd.read_csv(checkpoint_file)
        start_index = len(processed_repair)
        print(f"Checkpoint found. Resuming from index {start_index}.")
    else:
        processed_repair = pd.DataFrame(columns=repairs.columns.tolist())
        start_index = 0

    # Create a progress bar
    with tqdm(total=len(repairs), initial=start_index, desc="Processing repairs") as pbar:
        for index in range(start_index, len(repairs)):
            row = repairs.iloc[index]
            
            subject = row['subject']
            property = row['property']
            obj = row['object']
            
            if (index % 100000 == 0) and (index > start_index):
                print(f"Processing row {index}. Saving checkpoint.")
                # Save checkpoint
                processed_repair.to_csv(checkpoint_file, index=False)
            
            if (index % 10000 == 0):
                print(f"Processed {index} rows.")
            
            if obj.startswith('http'):
                # Call isRemovedWithObj function
                removed = isRemovedWithObj(subject, property, obj)
                if removed is not None:
                    # Update instanceRemoved column in repairs DataFrame
                    repairs.at[index, 'instanceRemoved'] = removed
                    # Create a new row for processed_repair DataFrame
                    new_row = repairs.loc[index].tolist()
                    processed_repair.loc[index] = new_row
                else:
                    print(f"Error in: {subject} {property} {obj}")

            # Update the progress bar
            pbar.update(1)

    # Save the final result
    processed_repair.to_csv(checkpoint_file, index=False)
    print("Processing complete. Final checkpoint saved.")

# Call the function to process the repairs
process_repairs(repairs)

In [ ]:
import pandas as pd
import requests
import xml.etree.ElementTree as ET
import os
import time
from tqdm import tqdm

def isRemovedWithObj(subject, property, obj, max_attempts=3):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"ASK {{ <{subject}> <{property}> <{obj}> }}"

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    for attempt in range(1, max_attempts + 1):
        response = requests.get(url, headers=headers)
        
        # Check if the request was successful and parse the response
        if response.ok:
            # Parse the XML response
            root = ET.fromstring(response.text)
            boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
            if boolean_element is not None:
                return boolean_element.text.lower() == 'false'
            else:
                print("Error: 'boolean' element not found in XML response")
                return None
        else:
            print(f"Error on attempt {attempt} for subject: {subject}, property: {property}, object: {obj}")
        
        if attempt < max_attempts:
            time.sleep(1)  # Optional: wait for 1 second before retrying
    
    return None

def process_repairs(repairs, checkpoint_file='symmetric/checkpoint.csv'):
    # Check if there is a checkpoint file and load it
    if os.path.exists(checkpoint_file):
        processed_repair = pd.read_csv(checkpoint_file)
        start_index = len(processed_repair)
        print(f"Checkpoint found. Resuming from index {start_index}.")
    else:
        processed_repair = pd.DataFrame(columns=repairs.columns.tolist())
        start_index = 0

    # Create a progress bar
    with tqdm(total=len(repairs), initial=start_index, desc="Processing repairs") as pbar:
        for index in range(start_index, len(repairs)):
            row = repairs.iloc[index]
            
            subject = row['subject']
            property = row['property']
            obj = row['object']
            
            if (index % 100000 == 0) and (index > start_index):
                print(f"Processing row {index}. Saving checkpoint.")
                # Save checkpoint
                processed_repair.to_csv(checkpoint_file, index=False)
            
            if (index % 10000 == 0):
                print(f"Processed {index} rows.")
            
            if obj.startswith('http'):
                # Call isRemovedWithObj function
                removed = isRemovedWithObj(subject, property, obj)
                if removed is not None:
                    # Update instanceRemoved column in repairs DataFrame
                    repairs.at[index, 'instanceRemoved'] = removed
                    # Create a new row for processed_repair DataFrame
                    new_row = repairs.loc[index].tolist()
                    processed_repair.loc[index] = new_row
                else:
                    print(f"Error in: {subject} {property} {obj}")

            # Update the progress bar
            pbar.update(1)

    # Save the final result
    processed_repair.to_csv(checkpoint_file, index=False)
    print("Processing complete. Final checkpoint saved.")

# Call the function to process the repairs
process_repairs(repairs)

In [ ]:
import pandas as pd
import requests
import xml.etree.ElementTree as ET
import os
import time
from tqdm import tqdm

def isRemovedWithObj(subject, property, obj, max_attempts=3):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"ASK {{ <{subject}> <{property}> <{obj}> }}"

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    for attempt in range(1, max_attempts + 1):
        response = requests.get(url, headers=headers)
        
        # Check if the request was successful and parse the response
        if response.ok:
            # Parse the XML response
            root = ET.fromstring(response.text)
            boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
            if boolean_element is not None:
                return boolean_element.text.lower() == 'false'
            else:
                print("Error: 'boolean' element not found in XML response")
                return None
        else:
            print(f"Error on attempt {attempt} for subject: {subject}, property: {property}, object: {obj}")
        
        if attempt < max_attempts:
            time.sleep(1)  # Optional: wait for 1 second before retrying
    
    return None

def process_repairs(repairs, checkpoint_file='symmetric/checkpoint.csv'):
    # Check if there is a checkpoint file and load it
    if os.path.exists(checkpoint_file):
        processed_repair = pd.read_csv(checkpoint_file)
        start_index = len(processed_repair)
        print(f"Checkpoint found. Resuming from index {start_index}.")
    else:
        processed_repair = pd.DataFrame(columns=repairs.columns.tolist())
        start_index = 0

    # Create a progress bar
    with tqdm(total=len(repairs), initial=start_index, desc="Processing repairs") as pbar:
        for index in range(start_index, len(repairs)):
            row = repairs.iloc[index]
            
            subject = row['subject']
            property = row['property']
            obj = row['object']
            
            if (index % 10000 == 0) and (index > start_index):
                print(f"Processing row {index}. Saving checkpoint.")
                # Save checkpoint
                processed_repair.to_csv(checkpoint_file, index=False)
            
            if (index % 10000 == 0):
                print(f"Processed {index} rows.")
            
            if obj.startswith('http'):
                # Call isRemovedWithObj function
                removed = isRemovedWithObj(subject, property, obj)
                if removed is not None:
                    # Update instanceRemoved column in repairs DataFrame
                    repairs.at[index, 'instanceRemoved'] = removed
                    # Create a new row for processed_repair DataFrame
                    new_row = repairs.loc[index].tolist()
                    processed_repair.loc[index] = new_row
                else:
                    print(f"Error in: {subject} {property} {obj}")

            # Update the progress bar
            pbar.update(1)

    # Save the final result
    processed_repair.to_csv(checkpoint_file, index=False)
    print("Processing complete. Final checkpoint saved.")

In [ ]:

# Call the function to process the repairs
process_repairs(repairs)

In [ ]:
import pandas as pd

# Read the first CSV file into a DataFrame
repairs = pd.read_csv("repairs_backup.csv")

repairs

In [ ]:
## 3515714 old
len(repairs[(repairs['instanceRemoved'] == True)])

In [ ]:
len(repairs[(repairs['instanceRemoved'] == True)])

In [ ]:
repairs['symmetricAdded'] = None

In [ ]:
repairs

In [ ]:
import requests
import xml.etree.ElementTree as ET

def symmetricAdded(subject, property, obj, max_attempts=3):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"ASK {{ <{obj}> <{property}> <{subject}> }}"

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    for attempt in range(1, max_attempts + 1):
        response = requests.get(url,headers=headers)

        # Check if the request was successful and parse the response
        if response.ok:
            # Parse the XML response
            #print(response.text)
            root = ET.fromstring(response.text)
            boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
            if boolean_element is not None:
                return boolean_element.text.lower() == 'true'
            else:
                print("Error: 'boolean' element not found in XML response")
                return None
        else:
            print(f"Error on attempt {attempt} for subject: {subject}, property: {property}, object: {obj}")
        
        if attempt < max_attempts:
            time.sleep(1)  # Optional: wait for 1 second before retrying
        
    return None
# Example usage
subject = "http://www.wikidata.org/entity/Q10925338"
property = "http://www.wikidata.org/prop/direct/P1322"
obj = "http://www.wikidata.org/entity/Q24840677"
print(symmetricAdded(subject, property, obj))

In [ ]:
for index, row in repairs.iterrows():
    # Extract subject and property without the prefix "http://www.wikidata.org/entity/"
    subject = row['subject']
    property = row['property']
    obj = row['object']
    
    if (index % 100000 == 0):
        print(index)
    #break
    
    if (obj.startswith('http') and row['instanceRemoved'] == False):
        # Call symmetricAdded function
        symm = symmetricAdded(subject, property, obj)

        # Update instanceRemoved column
        repairs.at[index, 'symmetricAdded'] = symm
    else:
        repairs.at[index, 'symmetricAdded'] = False

# Display the updated DataFrame
#print(filtered_df)

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import time
from tqdm import tqdm

# Your existing function
def symmetricAdded(subject, property, obj, max_attempts=3):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"ASK {{ <{obj}> <{property}> <{subject}> }}"

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    for attempt in range(1, max_attempts + 1):
        response = requests.get(url, headers=headers)

        # Check if the request was successful and parse the response
        if response.ok:
            # Parse the XML response
            root = ET.fromstring(response.text)
            boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
            if boolean_element is not None:
                return boolean_element.text.lower() == 'true'
            else:
                print("Error: 'boolean' element not found in XML response")
                return None
        else:
            print(f"Error on attempt {attempt} for subject: {subject}, property: {property}, object: {obj}")
        
        if attempt < max_attempts:
            time.sleep(1)  # Optional: wait for 1 second before retrying
    
    return None

# Function to save DataFrame periodically
def save_dataframe(df, filename="repairs_backup.csv"):
    df.to_csv(filename, index=False)

# Load the DataFrame (optional if you already have it in memory)
# repairs = pd.read_csv("your_repairs_file.csv")

# Process each row with a progress bar
for index, row in tqdm(repairs.iterrows(), total=repairs.shape[0], desc="Processing rows"):
    # Extract subject and property without the prefix "http://www.wikidata.org/entity/"
    subject = row['subject']
    property = row['property']
    obj = row['object']
    
    if pd.isna(row['symmetricAdded']):
        if (obj.startswith('http') and row['instanceRemoved'] == False):
            # Call symmetricAdded function
            symm = symmetricAdded(subject, property, obj)

            # Update symmetricAdded column
            repairs.at[index, 'symmetricAdded'] = symm
        else:
            repairs.at[index, 'symmetricAdded'] = False

    # Save the DataFrame every 50,000 rows
    #if (index + 1) % 50000 == 0:
    #    save_dataframe(repairs)
    #    print(f"Saved backup at row {index + 1}")

# Final save after the loop completes
save_dataframe(repairs)
print("Final save completed.")


In [ ]:
repairs

In [ ]:
repairs.to_csv("repairs_no_blank.csv", index=False)

In [ ]:
len(repairs[(repairs['symmetricAdded'] == True)])

In [ ]:
len(repairs[(repairs['symmetricAdded'] == True)])

In [ ]:
repairs[(repairs['symmetricAdded'] == False) & 
     (repairs['instanceRemoved'] == False) & 
     (repairs['exception'] == False)& 
     (repairs['deprecated rank'] == False)& 
     (repairs['deleted constraint'] == False)
    ]

In [ ]:
import requests
import xml.etree.ElementTree as ET

def isRemoved(subject, property):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"ASK {{ <{subject}> <{property}> [] }}"

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)

    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
subject = "http://www.wikidata.org/entity/Q456784"
property = "http://www.wikidata.org/prop/direct/P190"
print(isRemoved(subject, property))

In [ ]:
for index, row in repairs.iterrows():
    # Extract subject and property without the prefix "http://www.wikidata.org/entity/"
    if (index % 100000 == 0):
        print(index)
    
    if (row['object'].startswith('genid') and row['instanceRemoved'] == False and row['symmetricAdded'] == False):
        subject = row['subject']
        property = row['property']
        
        # Call isRemovedWithObj function
        removed = isRemoved(subject, property)

        # Update instanceRemoved column
        repairs.at[index, 'instanceRemoved'] = removed

# Display the updated DataFrame
#print(filtered_df)

In [ ]:
repairs[(repairs['symmetricAdded'] == False) & 
     (repairs['instanceRemoved'] == False)
    ]

In [ ]:
import pandas as pd

# Read the first CSV file into a DataFrame
repairs_left = pd.read_csv("symmetric_repairs.csv")

repairs_left

In [ ]:
import pandas as pd

# Assuming repairs_left and repairs are your DataFrames

# Select only the relevant columns for comparison
repairs_left_subset = repairs_left[['subject', 'property', 'object']]
repairs_subset = repairs[['subject', 'property', 'object']]

# Perform a merge with indicator to identify rows only in repairs_left
merged = repairs_left_subset.merge(repairs_subset, on=['subject', 'property', 'object'], how='left', indicator=True)

# Filter rows that are only in repairs_left
unique_to_repairs_left = merged[merged['_merge'] == 'left_only']

# Now, use the index of these rows to get the full rows from repairs_left
result = repairs_left.loc[unique_to_repairs_left.index]

In [ ]:
result

In [ ]:
result['instanceRemoved'] = None

In [ ]:
result['symmetricAdded'] = None

In [ ]:
for index, row in result.iterrows():
    # Extract subject and property without the prefix "http://www.wikidata.org/entity/"
    if (index % 100 == 0):
        print(index)
    
    if (row['instanceRemoved'] == None):
        subject = row['subject']
        property = row['property']
        
        # Call isRemovedWithObj function
        removed = isRemoved(subject, property)

        # Update instanceRemoved column
        result.at[index, 'instanceRemoved'] = removed

# Display the updated DataFrame
#print(filtered_df)

In [ ]:
result

In [ ]:
len(result[ 
     (result['instanceRemoved'] == True)
    ])

In [ ]:
result[ 
     (result['instanceRemoved'] == False)
    ]

In [ ]:
len(repairs)

In [ ]:
# Assuming repairs and result are your DataFrames

# Filter rows in result where instanceRemoved is True
result_true_instance_removed = result[result['instanceRemoved'] == True]

# Concatenate the filtered rows to the repairs DataFrame
repairs = pd.concat([repairs, result_true_instance_removed], ignore_index=True)

In [ ]:
len(repairs)

In [ ]:
repairs

In [ ]:
import numpy as np

# Assuming repairs is your DataFrame

# Filter rows where object starts with 'genid' and symmetricAdded is None
mask = repairs['object'].str.startswith('genid') & repairs['symmetricAdded'].isna()

# Set symmetricAdded to False for the filtered rows
repairs.loc[mask, 'symmetricAdded'] = False

In [ ]:
repairs

In [ ]:
len(repairs[ 
     (repairs['deleted constraint'] == True)
    ])

In [ ]:
len(repairs[ 
     (repairs['deprecated rank'] == True)
    ])

In [ ]:
len(repairs[ 
     (repairs['exception'] == True)
    ])

In [ ]:
len(repairs[ 
     (repairs['instanceRemoved'] == True)
    ])

In [ ]:
len(repairs[ 
     (repairs['symmetricAdded'] == True)
    ])

In [ ]:
repairs.dtypes

In [ ]:
repairs.to_csv("final_symmetric_repairs.csv", index=False)

In [ ]:
pip install matplotlib upsetplot

In [ ]:
pip install matplotlib-venn

In [ ]:
import pandas as pd
from matplotlib_venn import venn3
import matplotlib.pyplot as plt

# Load the repairs dataframe
# Assuming repairs is already loaded

# Extract the boolean columns
bool_columns = ['deleted constraint', 'deprecated rank', 'exception', 'instanceRemoved', 'symmetricAdded']

# For simplicity, let's visualize the first 3 boolean columns
subset = repairs[bool_columns[:3]]

# Calculate the set sizes and intersections
set_labels = bool_columns[:3]
sets = {
    set_labels[0]: set(subset.index[subset[set_labels[0]]]),
    set_labels[1]: set(subset.index[subset[set_labels[1]]]),
    set_labels[2]: set(subset.index[subset[set_labels[2]]]),
}

# Generate the Venn diagram
venn3([sets[set_labels[0]], sets[set_labels[1]], sets[set_labels[2]]],
      (set_labels[0], set_labels[1], set_labels[2]))

# Display the plot
plt.show()


In [ ]:
import pandas as pd

# Read the first CSV file into a DataFrame
repairs = pd.read_csv("final_symmetric_repairs.csv")

repairs

In [ ]:
repairs.dtypes

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

# Create the new DataFrame with the required columns
df2 = pd.DataFrame()
df2['A-box changes'] = repairs['instanceRemoved'] | repairs['symmetricAdded']
df2['T-box changes'] = (
    repairs['deleted constraint'] | 
    repairs['deprecated rank'] | 
    repairs['exception']
)

# Calculate the sizes of the sets
a_box_changes = df2['A-box changes'].sum()
t_box_changes = df2['T-box changes'].sum()
intersection = (df2['A-box changes'] & df2['T-box changes']).sum()

# Plot the Venn diagram
venn2(subsets=(a_box_changes, t_box_changes, intersection), 
      set_labels=('A-box changes', 'T-box changes'))


# Add title
plt.title("Symmetric Constraint share of repairs")

plt.show()


In [ ]:
repairs_P2293 =  repairs[(repairs['property'] == 'http://www.wikidata.org/entity/P2293')]

In [ ]:
repairs_P2293